Name: Viraj Bharat Pahade 
Student ID: 231049470

In [1]:
# ===============================================================
# Cell 1: Import libraries and basic setup
# ===============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import statsmodels.api as sm

# make folder for saving graphs & tables
SAVE_DIR = Path("figs_tables")
SAVE_DIR.mkdir(exist_ok=True)

# file with data
FILE = "cleaned data.xlsx"
SHEET = 0
DAYFIRST = True   # True if dates are in D/M/Y format

In [2]:
# ===============================================================
# Cell 2: Load and clean Bloomberg data
# ===============================================================
# read excel with 2 header rows and first column as date index
df_raw = pd.read_excel(FILE, sheet_name=SHEET, header=[0,1], index_col=0)
df_raw.index = pd.to_datetime(df_raw.index, errors="coerce", dayfirst=DAYFIRST)

# fix the headers (Bloomberg puts 'Unnamed' in many places)
lvl0 = pd.Series(df_raw.columns.get_level_values(0)).replace(r'^Unnamed.*', np.nan, regex=True).ffill()
lvl1 = pd.Series(df_raw.columns.get_level_values(1)).fillna('')
flat_cols = [f"{a}".strip() + (f"__{b}".strip() if str(b).strip()!='' else '') for a,b in zip(lvl0, lvl1)]
df_raw.columns = flat_cols

# convert to numbers and drop empty columns
num_df = df_raw.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")

C:\Users\aaa\AppData\Local\Temp\ipykernel_8404\1694030492.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_raw.index = pd.to_datetime(df_raw.index, errors="coerce", dayfirst=DAYFIRST)


In [3]:
# ===============================================================
# Cell 3: Select price and sentiment columns
# ===============================================================
PRICE_PATTERNS = ["__last price (usd)", "__last price", "__close", "__px_last"]
SENT_PATTERNS  = ["news sentiment - daily average", "sentiment"]

price_cols = [c for c in num_df.columns if any(p in c.lower() for p in PRICE_PATTERNS)]
sent_cols  = [c for c in num_df.columns if any(p in c.lower() for p in SENT_PATTERNS)]

# clean names (remove "Equity" etc.)
def short_name(b): return b.replace(" Equity","").strip()

prices = num_df[price_cols].copy()
prices.rename(columns={c: short_name(c.split("__")[0]) for c in prices.columns}, inplace=True)

sent = num_df[sent_cols].copy()
sent.rename(columns={c: short_name(c.split("__")[0])+"_Sent" for c in sent.columns}, inplace=True)


In [4]:
# ===============================================================
# Cell 4: Calculate returns and sentiment change
# ===============================================================
# daily percentage change for prices
returns = prices.pct_change(fill_method=None)

# daily difference for sentiment
sent_change = sent.diff()

# combine them for analysis
data_for_analysis = pd.concat([returns, sent_change], axis=1).dropna(how="all")
print("returns shape:", returns.shape, "| sentiment Δ shape:", sent_change.shape)

returns shape: (1828, 15) | sentiment Δ shape: (1828, 7)


In [5]:
# ===============================================================
# Cell 5: Descriptive statistics
# ===============================================================
desc_stats = data_for_analysis.agg(['mean','std','skew','kurtosis']).T
desc_stats.to_csv(SAVE_DIR / "table_descriptive_stats.csv")
print(desc_stats.head())

            mean       std      skew   kurtosis
DAL US  0.000408  0.026929 -0.145769  13.474950
AAL US -0.000003  0.034924  1.450490  20.069581
RYA ID  0.000416  0.023938  0.030716   6.070900
RYA ID  0.000355  0.024750  0.048839   5.536032
LHA GR -0.000387  0.024814  0.483016   6.221065


In [6]:
# ===============================================================
# Cell 6: Correlation heatmap
# ===============================================================
corr = data_for_analysis.corr()
corr.to_csv(SAVE_DIR / "table_correlation_matrix.csv")

fig, ax = plt.subplots(figsize=(10,8))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns))); ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticklabels(corr.index, fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label("Correlation", rotation=270, labelpad=15)
ax.set_title("Correlation Matrix: Airline Returns, Oil, Sentiment")
plt.tight_layout(); plt.savefig(SAVE_DIR / "fig_corr_heatmap.png", dpi=300); plt.close()

In [7]:
# ===============================================================
# Cell 7: Pick one airline and oil series
# ===============================================================
def pick_col(cols, hints):
    hints = [h.lower() for h in hints]
    for c in cols:
        if any(h in c.lower() for h in hints):
            return c
    return cols[0] if len(cols)>0 else None

airline_col = pick_col(returns.columns.tolist(), ["dal us","rya id","iag ln","lha gr","sia sp","qan au","aal us"])
oil_col     = pick_col(returns.columns.tolist(), ["jet","keros","brent","oil","gulf","singap","nwe"])

print("Chosen airline:", airline_col)
print("Chosen oil:", oil_col)

Chosen airline: DAL US
Chosen oil: DAL US


In [8]:
# ===============================================================
# Cell 8: Distribution of daily returns
# ===============================================================
x = returns[airline_col].dropna()
fig, ax = plt.subplots(figsize=(6,4))
ax.hist(x, bins=60, density=True, alpha=0.7)
ax.set_title(f"Distribution of {airline_col} Daily Returns")
ax.set_xlabel("Return"); ax.set_ylabel("Density")
plt.tight_layout(); plt.savefig(SAVE_DIR / "fig_distribution.png", dpi=300); plt.close()

In [9]:
# ===============================================================
# Cell 9: Rolling volatility
# ===============================================================
roll_vol = returns[airline_col].rolling(30).std()
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(roll_vol.index, roll_vol.values)
ax.set_title(f"30-Day Rolling Volatility: {airline_col}")
ax.set_xlabel("Date"); ax.set_ylabel("Volatility")
plt.tight_layout(); plt.savefig(SAVE_DIR / "fig_rolling_volatility.png", dpi=300); plt.close()

In [10]:
# ===============================================================
# Cell 10: Cumulative performance (airline vs oil)
# ===============================================================
cum_air = (1 + returns[airline_col].fillna(0)).cumprod()
cum_oil = (1 + returns[oil_col].fillna(0)).cumprod()

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(cum_air.index, cum_air.values, label=airline_col)
ax.plot(cum_oil.index, cum_oil.values, label=oil_col)
ax.legend(); ax.set_title(f"Cumulative Performance: {airline_col} vs {oil_col}")
ax.set_xlabel("Date"); ax.set_ylabel("Index (start=1)")
plt.tight_layout(); plt.savefig(SAVE_DIR / "fig_cumulative.png", dpi=300); plt.close()

In [11]:
# ===============================================================
# Cell 11: Rolling correlation between airline and oil
# ===============================================================
rolling_corr = returns[airline_col].rolling(60).corr(returns[oil_col])
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(rolling_corr.index, rolling_corr.values)
ax.set_title(f"Rolling 60-Day Correlation: {airline_col} vs {oil_col}")
ax.set_xlabel("Date"); ax.set_ylabel("Correlation")
plt.tight_layout(); plt.savefig(SAVE_DIR / "fig_rolling_corr.png", dpi=300); plt.close()

In [12]:
# ===============================================================
# Cell 12: Scatterplot with regression line (airline vs oil)
# ===============================================================
x = returns[oil_col]
y = returns[airline_col]
mask = x.notna() & y.notna()
x, y = x[mask], y[mask]

b, a = np.polyfit(x.values, y.values, 1)
x_line = np.linspace(x.min(), x.max(), 200)
y_line = a + b * x_line

fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(x.values, y.values, s=8, alpha=0.5)
ax.plot(x_line, y_line, color="red", linewidth=2)
ax.set_title(f"{airline_col} vs Oil Returns (fitted line)")
ax.set_xlabel(f"Δ{oil_col}"); ax.set_ylabel(f"{airline_col} return")
plt.tight_layout(); plt.savefig(SAVE_DIR / "fig_scatter.png", dpi=300); plt.close()

In [13]:
# ===============================================================
# Cell 13: Regression analysis (airline returns vs oil + sentiment)
# ===============================================================
X = pd.DataFrame(index=data_for_analysis.index)
X["dOil"] = returns[oil_col]

sent_cols_avail = [c for c in sent_change.columns if c.endswith("_Sent")]
if sent_cols_avail:
    X["dSent"] = sent_change[sent_cols_avail[0]]

y = returns[airline_col]
reg_df = pd.concat([y.rename("R_airline"), X], axis=1).dropna()

X_reg = sm.add_constant(reg_df[["dOil"] + (["dSent"] if "dSent" in reg_df.columns else [])])
y_reg = reg_df["R_airline"]

model = sm.OLS(y_reg, X_reg).fit()

coef_table = pd.DataFrame({
    "Variable": model.params.index,
    "Coefficient": model.params.values,
    "t-stat": model.tvalues.values,
    "p-value": model.pvalues.values
})
coef_table.to_csv(SAVE_DIR / "table_regression.csv", index=False)
print(coef_table)

  Variable   Coefficient        t-stat       p-value
0    const -3.794708e-19 -2.725230e+00  6.487110e-03
1     dOil  1.000000e+00  1.929913e+17  0.000000e+00
2    dSent -4.228388e-18 -5.458942e+00  5.446440e-08
